In [21]:
# Environment Setup and Imports


# Standard library
from pathlib import Path
import random
import time
import sys


# Numerical computing
import numpy as np
import pandas as pd


# Visualization
import matplotlib.pyplot as plt


# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim


# Computer vision
import torchvision
from torchvision import datasets, transforms, models


# Data loading
from torch.utils.data import DataLoader


# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


print("Environment initialized")

print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"Torchvision : {torchvision.__version__}")


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print(f"Device      : {device}")

if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")

Environment initialized
Python      : 3.12.3
PyTorch     : 2.12.1+cu130
Torchvision : 0.27.1+cu130
Device      : cuda
GPU         : NVIDIA GeForce RTX 5070 Ti


In [22]:
# Cell 2 — Reproducibility


SEED = 42


random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)


if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


print(f"Random seed fixed: {SEED}")

Random seed fixed: 42


In [26]:
# Cell 3 — Dataset Paths


datasetRoot = Path("../dataset")


trainPath = datasetRoot / "train"
valPath   = datasetRoot / "val"
testPath  = datasetRoot / "test"


print("Dataset exists:", datasetRoot.exists())

print("Train:", trainPath.exists())
print("Val  :", valPath.exists())
print("Test :", testPath.exists())

Dataset exists: True
Train: True
Val  : True
Test : True


In [33]:
# Cell 4 — Create PyTorch Datasets


trainDataset = datasets.ImageFolder(
    trainPath,
    transform=trainTransforms
)


valDataset = datasets.ImageFolder(
    valPath,
    transform=evalTransforms
)


testDataset = datasets.ImageFolder(
    testPath,
    transform=evalTransforms
)


classNames = trainDataset.classes
numClasses = len(classNames)


print(f"Classes ({numClasses}):")
print(classNames)

print("\nDataset sizes:")
print("Train:", len(trainDataset))
print("Val  :", len(valDataset))
print("Test :", len(testDataset))

Classes (8):
['AMD', 'CNV', 'CSR', 'DME', 'DR', 'DRUSEN', 'MH', 'NORMAL']

Dataset sizes:
Train: 18400
Val  : 2800
Test : 2800


In [35]:
# Cell 5 — DataLoaders


batchSize = 32


trainLoader = DataLoader(
    trainDataset,
    batch_size=batchSize,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)


valLoader = DataLoader(
    valDataset,
    batch_size=batchSize,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


testLoader = DataLoader(
    testDataset,
    batch_size=batchSize,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


print("DataLoaders ready")


images, labels = next(iter(trainLoader))

print("Image batch:", images.shape)
print("Label batch:", labels.shape)

DataLoaders ready
Image batch: torch.Size([32, 3, 224, 224])
Label batch: torch.Size([32])


In [37]:
# Cell 6 — ResNet50 Transfer Learning Model


model = models.resnet50(
    weights=models.ResNet50_Weights.IMAGENET1K_V2
)


# replace ImageNet classifier

inputFeatures = model.fc.in_features


model.fc = nn.Linear(
    inputFeatures,
    numClasses
)


model = model.to(device)


print("ResNet50 initialized")
print(model.fc)


totalParams = sum(
    p.numel() for p in model.parameters()
)


print(f"Total parameters: {totalParams:,}")

ResNet50 initialized
Linear(in_features=2048, out_features=8, bias=True)
Total parameters: 23,524,424


In [ ]:
# Cell 7 — Loss Function and Optimizer


criterion = nn.CrossEntropyLoss()


optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)


print("Training setup complete")

torch.Size([32, 8])


In [ ]:
def trainOneEpoch(model, dataLoader, criterion, optimizer, device):

    # Put model in training mode
    model.train()


    runningLoss = 0.0
    correctPredictions = 0
    totalSamples = 0


    for images, labels in dataLoader:

        # Move batch to device
        images = images.to(device)
        labels = labels.to(device)


        # Clear old gradients
        optimizer.zero_grad()


        # Forward pass
        outputs = model(images)


        # Calculate loss
        loss = criterion(outputs, labels)


        # Backpropagation
        loss.backward()


        # Update weights
        optimizer.step()


        # Track loss
        runningLoss += loss.item() * images.size(0)


        # Convert logits → predictions
        _, predictions = torch.max(outputs, 1)


        # Track accuracy
        correctPredictions += (
            predictions == labels
        ).sum().item()


        totalSamples += labels.size(0)



    epochLoss = runningLoss / totalSamples

    epochAccuracy = (
        correctPredictions / totalSamples
    )


    return epochLoss, epochAccuracy

In [ ]:
def validateModel(model, dataLoader, criterion, device):

    # Put model in evaluation mode
    model.eval()


    runningLoss = 0.0
    correctPredictions = 0
    totalSamples = 0


    # Disable gradient calculations
    with torch.no_grad():

        for images, labels in dataLoader:


            # Move batch to device
            images = images.to(device)
            labels = labels.to(device)


            # Forward pass
            outputs = model(images)


            # Calculate loss
            loss = criterion(outputs, labels)


            # Track loss
            runningLoss += loss.item() * images.size(0)


            # Convert logits to predictions
            _, predictions = torch.max(outputs, 1)


            # Track accuracy
            correctPredictions += (
                predictions == labels
            ).sum().item()


            totalSamples += labels.size(0)



    epochLoss = runningLoss / totalSamples


    epochAccuracy = (
        correctPredictions / totalSamples
    )


    return epochLoss, epochAccuracy

In [ ]:
# Training configuration

numEpochs = 5


history = {
    "trainLoss": [],
    "trainAccuracy": [],
    "valLoss": [],
    "valAccuracy": []
}


bestValAccuracy = 0.0


# Start training

for epoch in range(numEpochs):

    startTime = time.time()


    # Training phase

    trainLoss, trainAccuracy = trainOneEpoch(
        model,
        trainLoader,
        criterion,
        optimizer,
        device
    )


    # Validation phase

    valLoss, valAccuracy = validateModel(
        model,
        valLoader,
        criterion,
        device
    )


    # Store results

    history["trainLoss"].append(trainLoss)
    history["trainAccuracy"].append(trainAccuracy)

    history["valLoss"].append(valLoss)
    history["valAccuracy"].append(valAccuracy)



    # Save best model

    if valAccuracy > bestValAccuracy:

        bestValAccuracy = valAccuracy

        torch.save(
            model.state_dict(),
            "../models/bestResNet50.pth"
        )


    epochTime = time.time() - startTime



    print(
        f"Epoch [{epoch+1}/{numEpochs}] "
        f"| "
        f"Train Loss: {trainLoss:.4f} "
        f"| "
        f"Train Acc: {trainAccuracy:.4f} "
        f"| "
        f"Val Loss: {valLoss:.4f} "
        f"| "
        f"Val Acc: {valAccuracy:.4f} "
        f"| "
        f"Time: {epochTime:.1f}s"
    )


print("\nTraining complete")
print(f"Best validation accuracy: {bestValAccuracy:.4f}")

Epoch [1/5] | Train Loss: 0.3119 | Train Acc: 0.8997 | Val Loss: 0.1267 | Val Acc: 0.9561 | Time: 6241.3s


KeyboardInterrupt: 